# Chatbot with Conversational Memory

A custom chatbot built with LangChain and Groq, featuring persistent in-session memory and a Gradio chat interface. The assistant is pre-loaded with contextual user information and maintains full conversation history across turns.

In [ ]:
!pip install -U -q langchain_groq gradio

## Setup — LLM Client

I connect to the Groq API using the  model. The API key is loaded from Colab Secrets when running in Colab, or from a local  file otherwise.

In [ ]:
from langchain_groq import ChatGroq

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    import os
    load_dotenv()
    GROQ_API_KEY = os.getenv("GROQ_API_KEY")

llm_groq = ChatGroq(model_name="openai/gpt-oss-20b", api_key=GROQ_API_KEY)

### Quick connection test

In [ ]:
llm_groq.invoke("Hi")

## Chatbot Class

I implemented a simple memory mechanism by maintaining a running list of messages (system + user + assistant turns). Each new message is appended to the list before invoking the model, giving the LLM full conversation context. The system prompt defines the assistant persona and behavioral constraints.

In [ ]:
class Chatbot:
    prompt = """You are a helpful corporate assistant. Your name is Messan Komla!  Be polite and friendly. Use the memory provided to provide a
            factually correct clear information. Be concise and limit your responsed to less than 3 sentences."""

    def __init__(self):
      self.messages = [] # setting up a basic memory

      self.messages.append({"role": "system", "content": self.prompt})
      self.messages.append({"role": "user", "content": "This is basic information: User is living in San Francisco, California but is from Togo"})

    def __call__(self, user_message):
      self.messages.append({"role": "user", "content": user_message})
      ai_message = llm_groq.invoke(self.messages)
      self.messages.append({"role": "assistant", "content": ai_message.content})
      return ai_message.content + "\n" + "Tokens used: " + str(ai_message.response_metadata["token_usage"]["prompt_tokens"])

## Gradio Interface

I wrapped the chatbot in a Gradio  for an interactive browser-based UI.

In [ ]:
import gradio as gr
bot = Chatbot()
def chat(message, history):
    return bot(message)

demo = gr.ChatInterface(
    fn=chat,
    title="Gnona Chatbot",
    description="The great Chatbot with Meomory",
)
demo.launch(debug=True)

## Inspect Conversation Memory

I can inspect the full message history stored in the bot instance to verify memory is working correctly.

In [ ]:
bot.messages